# Video Food Classifier -- ViT with Temporal Pooling

This notebook walks through the full pipeline for classifying food videos as
**healthy** or **unhealthy** using the modular `vit_video` package.

Steps covered:
1. Environment check
2. Data download & frame extraction
3. Dataset inspection
4. Training
5. Evaluation on test split
6. Model validation (leakage audit, k-fold CV, external test)
7. Inference on a video file
8. Export to mobile formats
9. Upload to Hugging Face Hub

## 1. Environment Check

In [ ]:
import sys, importlib
from pathlib import Path

print(f"Python {sys.version}")

for pkg in ["torch", "torchvision", "timm", "cv2", "sklearn", "yt_dlp"]:
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, "__version__", "ok")
        print(f"  {pkg:16s} {ver}")
    except ImportError:
        print(f"  {pkg:16s} NOT INSTALLED")

import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Make sure imports resolve from the src/ directory
import _bootstrap
_bootstrap.setup()

from vit_video.utils.hardware import get_device, print_device_info

device = get_device()
print_device_info(hint_cuda=True)

## 2. Data Download & Frame Extraction

Downloads YouTube videos for each food category and extracts evenly-spaced
frames. Skips download if frames already exist.

In [ ]:
from vit_video.paths import DEFAULT_DATASET_DIR, DEFAULT_FRAMES_DIR
from vit_video.generatedata import load_categories, setup_folders, download_web_videos, extract_frames
from vit_video.data.splits import frames_directory_has_images

DATASET_DIR = DEFAULT_DATASET_DIR   # vit_video/food_data
FRAMES_DIR  = DEFAULT_FRAMES_DIR    # vit_video/food_data/frames

categories = load_categories(None)  # default healthy / unhealthy keywords
print(f"Categories: {list(categories.keys())}")
print(f"Total keywords: {sum(len(v) for v in categories.values())}")

In [ ]:
# Download videos and extract frames (skips if data already exists)
if not frames_directory_has_images(FRAMES_DIR):
    setup_folders(DATASET_DIR, categories)
    download_web_videos(DATASET_DIR, categories, num_videos_per_keyword=10)
    extract_frames(
        DATASET_DIR, categories,
        max_frames_per_video=60,
        frame_size=224,
        min_frames=1,
        num_workers=0,  # 0 = auto
    )
    print("Download and extraction complete.")
else:
    print(f"Frames already exist in {FRAMES_DIR} -- skipping download.")

## 3. Dataset Inspection

In [ ]:
from vit_video.data.splits import discover_videos_by_class, count_frame_images

videos_by_class = discover_videos_by_class(FRAMES_DIR)
total_frames = count_frame_images(FRAMES_DIR)

print(f"Total frames on disk: {total_frames}\n")
for cls, stems in sorted(videos_by_class.items()):
    print(f"  {cls}: {len(stems)} unique videos")

In [ ]:
# Visualise sample frames
import matplotlib.pyplot as plt
from vit_video.data.dataset import VideoDataset

preview_ds = VideoDataset(
    root=FRAMES_DIR,
    frames_per_video=8,
    img_size=224,
    augment=False,
)
print(f"Dataset size: {len(preview_ds)} videos, classes: {preview_ds.classes}")

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
video_tensor, label = preview_ds[0]
for i, ax in enumerate(axes.flat):
    frame = video_tensor[i].permute(1, 2, 0).numpy()
    frame = frame * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]  # un-normalise
    frame = frame.clip(0, 1)
    ax.imshow(frame)
    ax.set_title(f"Frame {i}")
    ax.axis("off")
fig.suptitle(f"Sample video -- class: {preview_ds.classes[label]}", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Training

Uses `train.py` via its Python API. Key defaults:
- Backbone: ViT-B/16 on GPU, MobileViT-XXS on CPU
- 25 epochs, batch 8, lr 3e-5, dropout 0.4
- AdamW + warmup (3 epochs) + cosine annealing
- Label smoothing 0.1, gradient clipping 1.0
- Class weighting enabled by default

In [ ]:
from vit_video.data.splits import ensure_split_manifest

# Create or reuse a video-level split manifest
manifest_path = ensure_split_manifest(
    frames_root=FRAMES_DIR,
    manifest_path=DATASET_DIR / "video_split_manifest.json",
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42,
)
print(f"Split manifest: {manifest_path}")

In [ ]:
import argparse
from vit_video.train import main as train_main

train_args = argparse.Namespace(
    dataset_dir=str(FRAMES_DIR),
    epochs=25,
    batch_size=8,
    lr=3e-5,
    weight_decay=1e-3,
    max_grad_norm=1.0,
    dropout=0.4,
    num_frames=8,
    img_size=224,
    disable_augmentation=False,
    class_weighting=True,
    min_samples_per_class=50,
    temporal_pool="avg",
    norm_mean="0.485,0.456,0.406",
    norm_std="0.229,0.224,0.225",
    hparam_search_epochs=0,
    lr_candidates="5e-6,1e-5,3e-5,5e-5,1e-4",
    num_workers=4,
    patience=7,
    min_delta=5e-5,
    backbone="auto",
    output_model="models/best_food_classifier.pth",
    resume_from="",
    split_manifest=str(manifest_path),
    no_auto_split_manifest=True,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    split_seed=42,
)

model_path = train_main(train_args)
print(f"\nSaved model: {model_path}")

### Plot training history

In [ ]:
import json
import matplotlib.pyplot as plt

history_file = Path(model_path).with_name(
    Path(model_path).stem + "_history.json"
)
if history_file.exists():
    history = json.loads(history_file.read_text())
else:
    print(f"History file not found: {history_file}")
    history = None

if history:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(history["train_loss"], label="Train", marker="o", markersize=4)
    ax1.plot(history["val_loss"], label="Val", marker="s", markersize=4)
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(history["train_acc"], label="Train", marker="o", markersize=4)
    ax2.plot(history["val_acc"], label="Val", marker="s", markersize=4)
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.set_title("Accuracy")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 5. Evaluation on Test Split

In [ ]:
from vit_video.test import build_test_loader, evaluate, print_results, save_results
from vit_video.utils.model_utils import load_model_from_checkpoint

model = load_model_from_checkpoint(model_path, device=device)

test_loader, classes = build_test_loader(
    dataset_dir=str(FRAMES_DIR),
    batch_size=4,
    num_frames=8,
    num_workers=0,
    split_manifest=str(manifest_path),
)

results = evaluate(model, test_loader, device, classes)
print_results(results)
save_results(results, "results")

In [ ]:
# Confusion matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

cm = np.array(results["confusion_matrix"])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.show()

print(f"\nAccuracy:  {results['accuracy']:.4f}")
print(f"Precision: {results['precision']:.4f}")
print(f"Recall:    {results['recall']:.4f}")
print(f"F1:        {results['f1']:.4f}")

## 6. Model Validation

Runs data-leakage audit, k-fold cross-validation, and external-data testing.

In [ ]:
# Run the full validation suite via CLI
import subprocess, sys

cmd = [
    sys.executable, "validate_model.py",
    "--model", str(model_path),
    "--dataset-dir", str(FRAMES_DIR),
    "--n-folds", "3",        # fewer folds for speed in notebook
    "--epochs", "5",         # shorter CV epochs
    "--skip-external",       # remove this flag to also test on fresh YouTube data
]
subprocess.run(cmd, check=False)

## 7. Inference on a Video File

In [ ]:
from vit_video.inference import predict_video

# Point this to any food video on disk
sample_video = Path(DATASET_DIR / "raw_videos" / "healthy")
mp4s = sorted(sample_video.glob("*.mp4"))[:1]

if mp4s:
    video_path = mp4s[0]
    pred_class, confidence, latency = predict_video(
        model, video_path, device,
        num_frames=8, img_size=224,
    )
    print(f"Video:      {video_path.name}")
    print(f"Prediction: {pred_class}")
    print(f"Confidence: {confidence:.2%}")
    print(f"Latency:    {latency:.0f} ms")
else:
    print("No sample videos found -- download data first (step 2).")

## 8. Export to Mobile Formats

Exports TorchScript and ONNX by default. Add `"coreml"` or `"tflite"` to the
formats list if the optional dependencies are installed.

In [ ]:
import subprocess, sys

export_cmd = [
    sys.executable, "export_mobile.py",
    "--model", str(model_path),
    "--output-dir", "exported_models",
    "--format", "torchscript", "onnx",
    "--eval-results", "results/test_results.json",
]
subprocess.run(export_cmd, check=False)

print("\nExported files:")
for f in sorted(Path("exported_models").glob("*")):
    size_mb = f.stat().st_size / 1024 / 1024 if f.is_file() else 0
    print(f"  {f.name:40s} {size_mb:>7.1f} MB" if f.is_file() else f"  {f.name}/")

## 9. Upload to Hugging Face Hub (Optional)

Requires `huggingface-hub` and a valid token (`huggingface-cli login`).

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, "upload_hf.py",
    "--repo-id", "maia2000/food-classifier",
    "--export-dir", "exported_models",
], check=False)

## Model Summary

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model Summary")
print(f"  Architecture:       MobileViTModel")
print(f"  Backbone:           {getattr(model, 'model_name', 'auto-detected')}")
print(f"  Temporal pooling:   {getattr(model, 'temporal_pool', 'avg')}")
print(f"  Total parameters:   {total_params:,}")
print(f"  Trainable params:   {trainable:,}")
print(f"  Input shape:        (B, 8, 3, 224, 224)")
print(f"  Output:             (B, 2)  [healthy, unhealthy]")
print(f"  Device:             {device}")